Tools


Models can request to call tools that perform tasks such as fetching data from a database, searching the web,or running code.
Tools are pairings of:
1) A schema(that includes the nam eof the tool,a description, and/or argument definitions) often a JSON schema.

2) A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model
#chat_models is a Python module.A module is simply a .py file (or a package/folder) that contains Python code such as classes, functions, and variables.
#chat_models is a package (a type of module) that groups together everything related to chat models.Classes (e.g., ChatOpenAI),Functions (e.g., init_chat_model),Other helper code.

#init_chat_model is a Python function.Its job is to create and return an instance (object) of the appropriate chat model.init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("Why do parrots talk?")
response


AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots and their ability to mimic human speech. Parrots are part of the Psittaciformes order, right? They\'re known for their vocal mimicry. But why do they do it?\n\nFirst, I think it\'s related to their communication. In the wild, parrots use vocalizations to interact with each other. Maybe they mimic human speech as a way to communicate with humans, similar to how they communicate with other parrots. But is that the main reason?\n\nI remember reading that parrots have a part of their brain called the song system, which helps with vocal learning. They also have a syrinx, which is the vocal organ in birds. The syrinx allows them to produce a wide range of sounds. So their anatomy supports their ability to mimic.\n\nAnother angle is social bonding. Parrots are social animals, so maybe they learn to talk to bond with their human companions. They might associate certain 

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get weather at a location"""
    return f"The weather at the {location} is sunny."

model_with_tools=model.bind_tools([get_weather])

In [ ]:
response=model_with_tools.invoke("What is the weather like in Banglore?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call["args"]}")


Tool Execution Loops

The model never runs the tool by itself,It only requests that a tool be run, the python code executes the tool and gives the result back to the model.

In [ ]:
tools={
    "get_weather":get_weather,
}

#The user asks
messages=[{"role":"user",
"content":"What's the weather in Chennai?"}]

ai_msg=model_with_tools.invoke(messages)
#AI Model requests to call get_weather(city="Chennai")

#Save the AI message
messages.append(ai_msg)

#Step: Execute the Tool
for tool_call in ai_msg.tool_calls:
    tool=tools[tool_call["name"]]
    tool_result=tool.invoke(tool_call["args"])
    messages.append(tool_result)

#Pass results back to model for final response
final_response=model_with_tools.invoke(messages)
print(final_response.text)
#The weather in Chennai is currently sunny. It's a great day to enjoy outdoor activities! ☀️

The weather in Chennai is currently sunny. It's a great day to enjoy outdoor activities! ☀️
